# Storage: CSV, gzip, and Parquet
**Before you begin:** Use Pure Python and attach `lh_pond` as the default Lakehouse.
<br>Run `0. Setup` with `DATA_SCOPE = "one_year"`.
<br>Run sections 1–4 in order; each uses paths or types defined earlier.
**Can stored files stay compact and queryable?**
<br>We’ll create plain CSV, gzip CSV, and Parquet copies of the same rows.
<br>Then compare their sizes and ask each copy the same SQL question.
Gzip (`.csv.gz`) compresses one CSV file. It is different from a ZIP archive.


## 1. Choose one month of flights
Use **January 2025** to keep the conversion manageable.
<br>The source arrives as Parquet; we’ll turn it into a CSV text export first.
**Conversion path:** source Parquet → plain CSV → gzip CSV and new Parquet.


In [ ]:
import duckdb
import os
from pathlib import Path

SOURCE = "/lakehouse/default/Files/flights_eu/one_year/flight_list_202501.parquet"
OUT = "/lakehouse/default/Files/storage_demo"
Path(OUT).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
rows = con.sql(f"SELECT count(*) FROM read_parquet('{SOURCE}')").fetchone()[0]
print(f"source : {SOURCE}")
print(f"rows   : {rows:,}")
print(f"on disk: {os.path.getsize(SOURCE)/1048576:.1f} MB  (Parquet, as delivered)")


## 1a. Carry the source types into CSV
**Parquet carries column types. CSV needs them inferred or supplied.**
<br>`DESCRIBE` gives us the source schema to supply when reading CSV.
This avoids type guessing, including for large `id` values.
<br>`column_types` is a Python dictionary: each column name maps to its SQL type.
<br>The loop builds this mapping from `DESCRIBE`; each CSV reader uses it.


In [ ]:
# Parquet carries its column names and types.
schema = con.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{SOURCE}')"
).fetchall()

# Build the name-to-type mapping for reading the CSV copies.
column_types = {}

print("columns:")
for name, data_type, *_ in schema:
    column_types[name] = data_type
    print(f"  {name:<22}{data_type}")


## 2. Write the same rows three ways
Write the plain CSV, then use it to create two compressed copies.
<br>Supply the saved types when reading the CSV.
- `.csv.gz` contains gzip-compressed CSV text.
- `.parquet` stores typed columns and uses Zstandard compression here.
All three outputs come from the same rows.
<br>`COPY (query) TO 'path'` writes a query result to a file.
<br>Run all three write cells before comparing the outputs.


In [ ]:
CSV = f"{OUT}/flights.csv"

con.execute(f"""
    COPY (SELECT * FROM read_parquet('{SOURCE}'))
    TO '{CSV}' (FORMAT CSV, HEADER)
""")
print("Wrote:", CSV)


### Gzip CSV
Compress the CSV text, keeping the source column types when reading it.


In [ ]:
GZIP = f"{OUT}/flights.csv.gz"

con.execute(f"""
    COPY (
        SELECT *
        FROM read_csv('{CSV}', header=true, types={column_types})
    )
    TO '{GZIP}' (FORMAT CSV, HEADER, COMPRESSION GZIP)
""")
print("Wrote:", GZIP)


### Parquet
Read the same CSV and write typed columns with Zstandard compression.


In [ ]:
PARQUET = f"{OUT}/flights.parquet"

con.execute(f"""
    COPY (
        SELECT *
        FROM read_csv('{CSV}', header=true, types={column_types})
    )
    TO '{PARQUET}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print("Wrote:", PARQUET)


## 2a. Check the copies
**CSV:** supply the saved column types.
<br>**Parquet:** read the types from the file itself.
All three copies should match the source count.
<br>Matching counts check row totals; they do not prove every value is identical.


In [ ]:
con.sql(f"""
    SELECT 'Source' AS format, count(*) AS rows
    FROM read_parquet('{SOURCE}')

    UNION ALL

    SELECT 'CSV', count(*)
    FROM read_csv('{CSV}', header=true, types={column_types})

    UNION ALL

    SELECT 'Gzip CSV', count(*)
    FROM read_csv('{GZIP}', header=true, types={column_types})

    UNION ALL

    SELECT 'Parquet', count(*)
    FROM read_parquet('{PARQUET}')
""").show()


## 3. Compare file sizes
**Same rows, different storage sizes.**
<br>`vs CSV` = plain CSV size ÷ this file’s size.
<br>For example, **5×** means one-fifth of the plain CSV size.
Python checks the file sizes; DuckDB handled the conversions.
<br>Compare the measured sizes. Smaller files do not automatically mean faster queries.


In [ ]:
files = {
    "CSV": CSV,
    "CSV.gz": GZIP,
    "Parquet (zstd)": PARQUET,
}
csv_size = os.path.getsize(CSV)
print(f"{'format':<20}{'MB':>10}{'vs CSV':>10}")
print("-" * 40)
for label, path in files.items():
    size = os.path.getsize(path)
    print(f"{label:<20}{size / 1048576:>10.1f}{csv_size / size:>9.1f}x")


## 4. Ask each file the same SQL question
**How many flights departed Barcelona (`LEBL`)?**
<br>Keep the SQL question; change only the reader in `FROM`.
All three counts should match.
<br>DuckDB reads each format directly, including gzip CSV.
<br>This compares answers, not query times.


In [ ]:
con.sql(f"""
    SELECT count(*) AS flights_from_barcelona
    FROM read_csv('{CSV}', header=true, types={column_types})
    WHERE adep = 'LEBL'
""").show()


### Gzip CSV
Use the same CSV reader with the compressed file.


In [ ]:
con.sql(f"""
    SELECT count(*) AS flights_from_barcelona
    FROM read_csv('{GZIP}', header=true, types={column_types})
    WHERE adep = 'LEBL'
""").show()


### Parquet
Change the reader. The file supplies its column types.


In [ ]:
con.sql(f"""
    SELECT count(*) AS flights_from_barcelona
    FROM read_parquet('{PARQUET}')
    WHERE adep = 'LEBL'
""").show()


## Optional: inspect the query plan
`EXPLAIN` shows how DuckDB plans the query.
<br>Look for **`READ_PARQUET`** and **`Filters: adep='LEBL'`** in the scan.
<br>The `~` row count is an estimate, not the count returned by the query.
<br>This shows the plan, not measured execution times.
<br>The preceding cell showed the answer. Set `RUN_EXPLAIN = True` to see its plan.


In [ ]:
RUN_EXPLAIN = False

if RUN_EXPLAIN:
    plan = con.sql(f"""
        EXPLAIN
        SELECT count(*) AS flights_from_barcelona
        FROM read_parquet('{PARQUET}')
        WHERE adep = 'LEBL'
    """).fetchone()[1]

    # Print the plan text so its diagram stays readable.
    print(plan)
else:
    print("Query plan skipped. Set RUN_EXPLAIN = True to display it.")


## Takeaway
> Convert once. Query again.
**Gzip CSV:** compressed text that DuckDB can read directly.
<br>**Parquet:** stored column types and selective column reads.
Use the measured sizes above to compare storage savings for this data.
Keep the original files when you need an archive.
<br>Keep a Parquet copy for repeated SQL analysis.


## Inspect or reset the demo folder
Run the cell to list files and sizes in `Files/storage_demo`.
**Optional reset:** set `RESET_DEMO_OUTPUT = True` to delete the folder and all its contents.
<br>Leave it `False` to list the files without deleting them.
<br>After a reset, rerun from section 1 to recreate the demo outputs.
<br>Return the flag to `False` before using **Run all** again.


In [ ]:
from pathlib import Path

OUT = Path("/lakehouse/default/Files/storage_demo")

RESET_DEMO_OUTPUT = False

if RESET_DEMO_OUTPUT:
    import shutil

    # Confirm the resolved target is the demo output folder before deleting it.
    expected_folder = Path("/lakehouse/default/Files").resolve() / "storage_demo"
    if OUT.resolve() != expected_folder:
        raise ValueError("Reset is restricted to Files/storage_demo.")

    if OUT.exists():
        shutil.rmtree(OUT)
        print("Deleted demo output folder:", OUT)

if OUT.exists():
    print("files in", OUT)
    for f in sorted(OUT.iterdir()):
        if f.is_file():
            print(f"  {f.name:<28}{f.stat().st_size / 1048576:>9.1f} MB")
else:
    print("No storage_demo folder. Run sections 1 and 2 to create it.")
